In [24]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sksurv.metrics import (
    as_concordance_index_ipcw_scorer,
    as_cumulative_dynamic_auc_scorer,
    as_integrated_brier_score_scorer,
)
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval

In [26]:
# Load formatted cytokines data
df = pd.read_csv("../data/CV_plasma_Combined_formatted_20260731.csv")
df

,sample_names,sample_id,cohort,egf,ifnalpha2,ifngamma,il1alpha,il1beta,il4,il6,il8,il10,mip1alpha,tgfalpha,tnfalpha,tnfbeta,hbegf,vegfc,vegfd,ages,sex,race,tobaccouse,packyears,alcoholuse,averageucsf,spontaneouspain135,functionalpain246,bpiphysicalinterferences,bpipsychologicalinterferences,bpi_intensity,functionalscores,symptomscore,globalhealthstatusql2,qlq30score,hn35,pathologicstage,lvi,pni,survival_status,sx_to_lastfu,survival_2yrs,ss
0,FF301C,FF301C,LLU,29.45,25.24,90.00,6.40,3.95,4.21,3.54,4.14,3.92,25.93,3.32,18.74,10.46,83.27,381.47,263.05,67,female,white_nonhispanic,never,0.0,never,1.50,0.33,0.33,0.0,0.0,0.0,83.33,19.44,75.00,81.62,30.56,stage_I,no,no,alive,1839,alive,1
1,FF302C,FF302C,LLU,71.50,9.93,46.35,0.00,0.00,2.33,3.42,4.53,75.20,2.90,1.52,25.54,3.74,75.20,923.75,237.53,68,male,white_nonhispanic,former,52.5,former,68.75,36.67,80.00,21.0,32.0,23.0,54.33,59.03,16.67,46.11,47.78,stage_IVA,no,yes,alive,1721,alive,4
2,FF303C,FF303C,LLU,54.54,18.38,54.01,46.49,18.90,1.37,4.15,3.57,98.33,25.05,2.99,22.39,4.83,51.96,490.19,130.47,78,male,white_nonhispanic,never,0.0,current,67.50,63.33,63.33,9.0,4.0,22.0,39.67,31.25,50.00,57.56,45.68,stage_IVA,yes,no,death,48,death,4
3,FF304C,FF304C,LLU,132.67,27.47,81.82,6.89,3.95,4.56,9.69,7.29,20.63,20.32,4.31,28.18,9.32,85.47,1148.22,34.22,49,male,white_nonhispanic,current,72.0,current,95.00,86.67,100.00,0.0,0.0,29.0,0.00,68.75,0.00,19.23,94.44,stage_IVA,no,yes,death,131,death,3
4,FF305C,FF305C,LLU,28.01,13.63,66.20,5.44,0.92,3.04,2.42,5.44,10.63,17.14,3.98,39.61,5.94,71.54,1113.45,142.55,65,female,white_nonhispanic,former,42.0,never,37.00,11.00,33.67,2.0,10.0,18.0,82.67,27.08,75.00,76.67,17.44,stage_II,no,no,alive,1679,alive,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,76,S0076,UAB,61.90,122.36,103.74,552.31,15.62,5864.00,199.75,208.34,168.84,18.23,NaN,19.66,855.53,NaN,289.94,91.56,60,female,white_nonhispanic,never,NaN,never,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,stage_IVB,yes,yes,death,882,alive,3
208,77,S0077,UAB,18.86,91.86,71.49,22.67,5.42,146.71,18.96,12.67,82.12,10.90,NaN,26.21,19.05,NaN,234.20,65.32,74,female,white_nonhispanic,never,NaN,never,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,stage_II,no,no,alive,1273,alive,1
209,78,S0078,UAB,12.99,66.46,51.61,9.46,2.89,28.61,2.28,8.89,7.41,9.54,NaN,13.77,7.24,NaN,144.61,55.09,60,male,white_nonhispanic,former,NaN,never,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,stage_III,yes,yes,alive,1250,alive,4
210,79,S0079,UAB,13.66,75.02,40.79,8.96,3.81,38.38,5.04,4.50,12.45,6.71,NaN,16.65,11.06,NaN,158.81,21.47,71,male,white_nonhispanic,current,NaN,current,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,stage_II,no,no,alive,1183,alive,2


In [27]:
# Encode survival status as binary variable
df["survival_status_encoded"] = df["survival_status"].map({"alive": 0, "death": 1})

In [28]:
# Map string labels to boolean: True = death observed, False = alive
event_observed = df["survival_status"].map(
    {"death": True, "alive": False}
)

# Check for NA values in survival_status mapping
assert event_observed.isna().sum() == 0, (
    f"Unmapped survival_status values found: "
    f"{df['survival_status'][event_observed.isna()].unique()}"
)

# Create structured array for survival analysis
y = Surv.from_arrays(
        event=event_observed,
        time=df["sx_to_lastfu"]
    )

y

array([(False, 1839.), (False, 1721.), ( True,   48.), ( True,  131.),
       (False, 1679.), ( True,  318.), ( True,  278.), (False, 1765.),
       (False, 1758.), ( True,  997.), ( True,  482.), ( True,  592.),
       (False, 1625.), (False, 1592.), (False, 1406.), ( True,   84.),
       ( True,  253.), ( True,   51.), (False, 1582.), (False, 1510.),
       (False, 1500.), (False, 1485.), (False, 1410.), (False, 1475.),
       ( True,  689.), ( True, 1055.), ( True,  433.), (False,  634.),
       ( True,  665.), (False, 1268.), ( True,  330.), (False, 1274.),
       (False, 1282.), (False,  765.), (False, 1123.), (False, 1094.),
       ( True,  618.), ( True,  490.), (False, 1137.), (False, 1095.),
       (False, 1050.), ( True,  494.), ( True,  597.), (False, 1085.),
       ( True,  379.), ( True,  355.), (False, 1078.), ( True,  385.),
       ( True,   87.), ( True,  763.), (False,  907.), (False,  814.),
       (False, 1018.), (False, 1014.), (False,  945.), (False,  631.),
      

In [29]:
# Specify lists of variables for analysis
clinicopathologic_vars = [
    "ages", "sex", "race", "tobaccouse", "alcoholuse",
    "pathologicstage", "lvi", "pni",
    "survival_status", "sx_to_lastfu", "survival_2yrs"
]
pain_scores = [
    "averageucsf", "spontaneouspain135", "functionalpain246", "bpiphysicalinterferences",
    "bpipsychologicalinterferences", "bpi_intensity", "functionalscores", "symptomscore",
    "globalhealthstatusql2", "qlq30score", "hn35"
]
cytokines = [
    "egf", "ifnalpha2",	"ifngamma",	"il1alpha",
    "il1beta", "il4", "il6", "il8",	"il10",
    "mip1alpha", "tgfalpha", "tnfalpha", "tnfbeta",
    "hbegf", "vegfc", "vegfd"
]

In [30]:
# Subset cytokines and cohort columns for analysis
cytokines_df = df.loc[:, cytokines + ["cohort", "ss", "survival_status_encoded"]].drop(columns=["tgfalpha", "hbegf"])
cytokines_df

,egf,ifnalpha2,ifngamma,il1alpha,il1beta,il4,il6,il8,il10,mip1alpha,tnfalpha,tnfbeta,vegfc,vegfd,cohort,ss,survival_status_encoded
0,29.45,25.24,90.00,6.40,3.95,4.21,3.54,4.14,3.92,25.93,18.74,10.46,381.47,263.05,LLU,1,0
1,71.50,9.93,46.35,0.00,0.00,2.33,3.42,4.53,75.20,2.90,25.54,3.74,923.75,237.53,LLU,4,0
2,54.54,18.38,54.01,46.49,18.90,1.37,4.15,3.57,98.33,25.05,22.39,4.83,490.19,130.47,LLU,4,1
3,132.67,27.47,81.82,6.89,3.95,4.56,9.69,7.29,20.63,20.32,28.18,9.32,1148.22,34.22,LLU,3,1
4,28.01,13.63,66.20,5.44,0.92,3.04,2.42,5.44,10.63,17.14,39.61,5.94,1113.45,142.55,LLU,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
207,61.90,122.36,103.74,552.31,15.62,5864.00,199.75,208.34,168.84,18.23,19.66,855.53,289.94,91.56,UAB,3,1
208,18.86,91.86,71.49,22.67,5.42,146.71,18.96,12.67,82.12,10.90,26.21,19.05,234.20,65.32,UAB,1,0
209,12.99,66.46,51.61,9.46,2.89,28.61,2.28,8.89,7.41,9.54,13.77,7.24,144.61,55.09,UAB,4,0
210,13.66,75.02,40.79,8.96,3.81,38.38,5.04,4.50,12.45,6.71,16.65,11.06,158.81,21.47,UAB,2,0


In [31]:
# Specify numeric features for ML models
numeric_features = cytokines_df.columns.drop(["cohort", "survival_status_encoded"])

In [32]:
# Best subset of features for RSF model
selected_features = numeric_features.intersection(["il6", "il10", "tnfalpha", "ss"])
print(f"Selected features for ML model: {selected_features}")

feature_cols = selected_features.tolist() + ["cohort"]

Selected features for ML model: Index(['il6', 'il10', 'tnfalpha', 'ss'], dtype='object')


In [33]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    cytokines_df[feature_cols],
    y,
    test_size=0.2,
    random_state=2024,
    stratify=y["event"],
)

In [66]:
# Sanity check: Ensure that the time range of the test data is within the time range of the training data
y_events = y_train[y_train["event"]]
train_min, train_max = y_events["time"].min(), y_events["time"].max()

y_events = y_test[y_test["event"]]
test_min, test_max = y_events["time"].min(), y_events["time"].max()

assert (
    train_min <= test_min < test_max < train_max
), "time range or test data is not within time range of training data."

In [45]:
# Wrap the cohort-aware scaling as an sklearn transformer
class CohortScaler(BaseEstimator, TransformerMixin):
    """StandardScaler fit independently within each 'cohort' group."""
    def __init__(self, feature_cols):
        self.feature_cols = feature_cols

    def fit(self, X, y=None):
        self.scalers_ = {}
        for cohort, group in X.groupby("cohort"):
            sc = StandardScaler()
            sc.fit(group[self.feature_cols])
            self.scalers_[cohort] = sc
        return self

    def transform(self, X):
        X = X.copy()
        parts = []
        for cohort, group in X.groupby("cohort"):
            sc = self.scalers_[cohort]  # assumes cohort seen in fit
            transformed = sc.transform(group[self.feature_cols])
            parts.append(pd.DataFrame(transformed, columns=self.feature_cols, index=group.index))
        return pd.concat(parts).loc[X.index]

In [59]:
# Wrap RSF with the metric of choice
# Specify times grid for cumulative_dynamic_auc call
lower, upper = np.percentile(y["time"], [10, 90])
eval_times = np.arange(lower, upper + 1)
# eval_times = times

def build_pipeline(params, metric="c_index"):
    rsf = RandomSurvivalForest(random_state=2026, n_jobs=-1, **params)

    if metric == "c_index":
        wrapped = as_concordance_index_ipcw_scorer(rsf, tau=eval_times[-1])
    elif metric == "auc":
        wrapped = as_cumulative_dynamic_auc_scorer(rsf, times=eval_times)
    elif metric == "brier":
        wrapped = as_integrated_brier_score_scorer(rsf, times=eval_times)
    else:
        raise ValueError(f"Unknown metric: {metric}")

    return Pipeline([
        ("scaler", CohortScaler(feature_cols=selected_features)),
        ("model", wrapped),
    ])

In [60]:
# Specify hyperopt objective using sksurv metrics
# Brier score is "lower is better"; concordance and AUC are "higher is better"
LOWER_IS_BETTER = {"c_index": False, "auc": False, "brier": True}

def make_objective(metric="c_index"):
    lower_better = LOWER_IS_BETTER[metric]

    def objective(params):
        params = {
            "n_estimators": int(params["n_estimators"]),
            "max_depth": int(params["max_depth"]),
            "min_samples_leaf": int(params["min_samples_leaf"]),
            "min_samples_split": int(params["min_samples_split"]),
            "max_features": params["max_features"],
        }

        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
        scores = []

        for train_idx, val_idx in cv.split(X_train, y_train["event"]):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            pipe = build_pipeline(params, metric=metric)
            pipe.fit(X_tr, y_tr)
            scores.append(pipe.score(X_val, y_val))

        mean_score = np.mean(scores)
        loss = mean_score if lower_better else -mean_score
        return {"loss": loss, "status": STATUS_OK}

    return objective

In [61]:
# Define the search space
space = {
    "n_estimators": hp.quniform("n_estimators", 50, 500, 25),
    "max_depth": hp.quniform("max_depth", 2, 12, 1),
    "min_samples_leaf": hp.quniform("min_samples_leaf", 3, 30, 1),
    "min_samples_split": hp.quniform("min_samples_split", 2, 20, 1),
    "max_features": hp.choice("max_features", ["sqrt", "log2", None]),
}

In [82]:
# Run the search for whichever metric
trials = Trials()

objective = make_objective(metric="auc")  # or ["c_index", "auc", "brier"]

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
    rstate=np.random.default_rng(2026),
)

best_params = space_eval(space, best)
best_params = {
    "n_estimators": int(best_params["n_estimators"]),
    "max_depth": int(best_params["max_depth"]),
    "min_samples_leaf": int(best_params["min_samples_leaf"]),
    "min_samples_split": int(best_params["min_samples_split"]),
    "max_features": best_params["max_features"],
}
print("Best params:", best_params)

100%|██████████| 50/50 [01:11<00:00,  1.42s/trial, best loss: -0.7867209748795339]
Best params: {'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 28, 'min_samples_split': 10, 'max_features': None}


In [ ]:
# c_index: Best params: {'n_estimators': 150, 'max_depth': 4, 'min_samples_leaf': 25, 'min_samples_split': 10, 'max_features': 'log2'}
# auc: Best params: {'n_estimators': 200, 'max_depth': 5, 'min_samples_leaf': 28, 'min_samples_split': 10, 'max_features': None}


In [85]:
from sksurv.metrics import cumulative_dynamic_auc

final_pipe = build_pipeline(best_params, metric="auc")
final_pipe.fit(X_train, y_train)
test_score = final_pipe.score(X_test, y_test)

risk_scores = final_pipe.predict(X_test)
auc, mean_auc = cumulative_dynamic_auc(y_train, y_test, risk_scores, eval_times)

print(f"C-index: {test_score:.3f}")
print(f"Mean AUC over time: {mean_auc:.3f}")

# Access the underlying fitted RSF if you want predict_survival_function etc.
rsf_tuned = final_pipe.named_steps["model"].estimator_

C-index: 0.813
Mean AUC over time: 0.813
